# Clustered Curve Beta CDF Model

This notebook consumes the CSV produced by `custpaydetails_clustered_cumulative_curves.sql`, fits Beta CDF expected cumulative burn curves, evaluates held-out items, and compares the Beta CDF approach to a pure linear spend model.

## Input and Parameterization

| Metric | Value |
| --- | --- |
| Input CSV | clustered_data_input_udot.csv |
| Cluster curve rows | 7,631 |
| Items | 875 |
| Train items | 716 |
| Test items | 159 |
| Global Beta alpha | 0.500202 |
| Global Beta beta | 0.776171 |
| Global train MSE | 0.04320586 |
| Proxy label CSV | clustered_curve_proxy_labels_udot.csv |
| Proxy-labeled held-out rows | 1,333 |

In [ ]:
import csv
from pathlib import Path

path = Path('clustered_data_input.csv')
if not path.exists():
    path = Path('custpaydetails_clustered_cumulative_curves.csv')
if not path.exists():
    path = Path('ci_item_clustered_cumulative_curves.csv')
with path.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
print(path)
print(len(rows))
print(reader.fieldnames)

## Linear Spend Baseline

The pure linear model assumes cumulative spend should equal elapsed time:

```text
expected_cumulative_pct_linear = elapsed_pct
position_delta_linear = actual_cumulative_pct - elapsed_pct
```

This is a useful baseline because it is transparent and easy to explain. It is also too rigid: many items are naturally front-loaded or back-loaded, so a linear curve can systematically flag normal burn shapes as risk.

## Held-Out Model Performance

Errors are cumulative-spend-percent errors. MAE `0.15` means an average absolute error of about 15 percentage points of final item spend.

| Model | MAE | RMSE | Median AE | P90 AE | Bias |
| --- | --- | --- | --- | --- | --- |
| Duration-bucket Beta CDF | 0.1394 | 0.1940 | 0.1018 | 0.3272 | 0.0054 |
| Cluster-count-bucket Beta CDF | 0.1455 | 0.2011 | 0.1101 | 0.3584 | -0.0008 |
| Pooled empirical median curve | 0.1472 | 0.2107 | 0.1011 | 0.3579 | 0.0022 |
| Global Beta CDF | 0.1476 | 0.2060 | 0.1073 | 0.3566 | 0.0041 |
| Linear cumulative spend | 0.1760 | 0.2454 | 0.1298 | 0.4106 | -0.1011 |

## Beta CDF vs Linear: Improvement Summary

| Comparison | Value |
| --- | --- |
| MAE improvement vs linear | 20.80% |
| RMSE improvement vs linear | 20.96% |
| P90 absolute-error improvement vs linear | 20.32% |
| Linear bias | -0.1011 |
| Duration-bucket Beta bias | 0.0054 |

## Error Comparison Plot



## Bucketed Beta CDF Parameters

### Duration Buckets

| Duration bucket | alpha | beta |
| --- | --- | --- |
| 181-365d | 0.526807 | 0.866044 |
| 366-730d | 0.546996 | 0.955521 |
| <=180d | 0.768117 | 0.690976 |
| >730d | 0.599462 | 1.179298 |

### Cluster-Count Buckets

| Cluster-count bucket | alpha | beta |
| --- | --- | --- |
| 13+ clusters | 0.726607 | 1.198410 |
| 4-6 clusters | 0.222490 | 0.295120 |
| 7-12 clusters | 0.459223 | 0.751965 |
| <=3 clusters | 0.143171 | 0.161213 |

## Curve Performance Plot



## Risk Signal Framing

For active items, the same fitted curve can produce budget-overrun and time-overrun risk signals. These are not final labels without a real authorized budget and schedule; they are position signals.

```text
expected_pct = model(elapsed_pct)
position_delta = actual_cumulative_pct - expected_pct

if position_delta > threshold:
    spending too quickly / budget-overrun risk

if position_delta < -threshold:
    spending too slowly / time-overrun risk
```

For budget overrun projection once a budget is available:

```text
projected_final_spend = actual_cumulative_spend / expected_pct
projected_overrun = projected_final_spend - authorized_budget
```

## Beta vs Linear Risk Signals on Held-Out Data

The table shows how many clustered observations would be flagged by each approach at several position-delta thresholds. `LinearOnly` rows are especially important: these are cases the linear model flags but the duration-bucket Beta curve treats as normal for the observed burn shape.

| Threshold | Signal | Beta count | Linear count | Both | Linear only | Beta only |
| --- | --- | --- | --- | --- | --- | --- |
| 0.10 | spending too quickly / budget-overrun risk | 307 | 587 | 292 | 295 | 15 |
| 0.10 | spending too slowly / time-overrun risk | 365 | 177 | 166 | 11 | 199 |
| 0.15 | spending too quickly / budget-overrun risk | 229 | 469 | 215 | 254 | 14 |
| 0.15 | spending too slowly / time-overrun risk | 279 | 134 | 128 | 6 | 151 |
| 0.25 | spending too quickly / budget-overrun risk | 124 | 295 | 119 | 176 | 5 |
| 0.25 | spending too slowly / time-overrun risk | 122 | 66 | 59 | 7 | 63 |

## Proxy ROC/AUC Setup

True ROC/AUC requires true binary outcome labels. This notebook now consumes retrospective proxy labels from `clustered_curve_proxy_labels.csv`, which is generated by the separate polynomial proxy-label notebook. The Beta CDF and linear models do not create those labels; they only score against them.

These ROC curves compare how well Beta and linear position scores recover the external proxy labels. They are useful for model behavior comparison, but they are not a substitute for true budget/schedule outcome labels.

## Proxy ROC/AUC Summary

| Signal | Model | AUC | Positive labels | Negative labels |
| --- | --- | --- | --- | --- |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.9860 | 254 | 1,079 |
| fast_spend_proxy | Linear cumulative spend | 0.9835 | 254 | 1,079 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.9797 | 273 | 1,060 |
| slow_spend_proxy | Linear cumulative spend | 0.9647 | 273 | 1,060 |

## Threshold Sweep for Proxy Labels

| Signal | Model | Threshold | Flagged | TP | FP | Precision | Recall/TPR | FPR |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.05 | 414 | 254 | 160 | 0.614 | 1.000 | 0.148 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.10 | 307 | 240 | 67 | 0.782 | 0.945 | 0.062 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.15 | 229 | 201 | 28 | 0.878 | 0.791 | 0.026 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.20 | 169 | 158 | 11 | 0.935 | 0.622 | 0.010 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.25 | 124 | 120 | 4 | 0.968 | 0.472 | 0.004 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.30 | 88 | 88 | 0 | 1.000 | 0.346 | 0.000 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.40 | 48 | 48 | 0 | 1.000 | 0.189 | 0.000 |
| fast_spend_proxy | Linear cumulative spend | 0.05 | 710 | 254 | 456 | 0.358 | 1.000 | 0.423 |
| fast_spend_proxy | Linear cumulative spend | 0.10 | 587 | 254 | 333 | 0.433 | 1.000 | 0.309 |
| fast_spend_proxy | Linear cumulative spend | 0.15 | 469 | 254 | 215 | 0.542 | 1.000 | 0.199 |
| fast_spend_proxy | Linear cumulative spend | 0.20 | 367 | 247 | 120 | 0.673 | 0.972 | 0.111 |
| fast_spend_proxy | Linear cumulative spend | 0.25 | 295 | 223 | 72 | 0.756 | 0.878 | 0.067 |
| fast_spend_proxy | Linear cumulative spend | 0.30 | 231 | 204 | 27 | 0.883 | 0.803 | 0.025 |
| fast_spend_proxy | Linear cumulative spend | 0.40 | 134 | 134 | 0 | 1.000 | 0.528 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.05 | 496 | 266 | 230 | 0.536 | 0.974 | 0.217 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.10 | 365 | 254 | 111 | 0.696 | 0.930 | 0.105 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.15 | 279 | 242 | 37 | 0.867 | 0.886 | 0.035 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.20 | 199 | 194 | 5 | 0.975 | 0.711 | 0.005 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.25 | 122 | 122 | 0 | 1.000 | 0.447 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.30 | 71 | 71 | 0 | 1.000 | 0.260 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.40 | 25 | 25 | 0 | 1.000 | 0.092 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.05 | 242 | 205 | 37 | 0.847 | 0.751 | 0.035 |
| slow_spend_proxy | Linear cumulative spend | 0.10 | 177 | 170 | 7 | 0.960 | 0.623 | 0.007 |
| slow_spend_proxy | Linear cumulative spend | 0.15 | 134 | 134 | 0 | 1.000 | 0.491 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.20 | 94 | 94 | 0 | 1.000 | 0.344 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.25 | 66 | 66 | 0 | 1.000 | 0.242 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.30 | 47 | 47 | 0 | 1.000 | 0.172 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.40 | 18 | 18 | 0 | 1.000 | 0.066 | 0.000 |

## Proxy ROC Curves



## Residual Distribution Plot

Residual is `actual cumulative pct - expected cumulative pct`. Positive residuals indicate spending ahead of expected position; negative residuals indicate spending behind expected position.



## Recommendations

Use the duration-bucket Beta CDF as the first expected-position model and keep the linear model as a transparent benchmark. For production alerting:

- Use Beta CDF `position_delta` for primary spend-too-fast and spend-too-slow signals.
- Show the linear delta as a secondary reference because users understand it.
- Alert only when position delta exceeds a threshold and remains elevated across updates.
- For budget-overrun detection, combine position delta with authorized budget and projected final spend.
- For time-overrun detection, combine slow-spend position delta with schedule metadata; slow spending can mean delay, scope removal, or inactive work.